# Đánh giá độ dài segment trên toàn bộ folder

Mỗi segment được hiểu theo dạng `path|passage`: phần trước dấu `|` là `path`; nếu không có dấu `|` thì `path` là chuỗi rỗng.

Notebook này quét **đệ quy toàn bộ folder** chứa processed context/segments, lấy từng segment và thống kê **độ dài theo số ký tự**, đồng thời tính **độ dài path** (phần nằm trước dấu `|`).

Hỗ trợ các JSON có cấu trúc phổ biến như:
- `{"passage_segments": ["...", "..."]}`
- `{"segments": ["...", "..."]}`
- danh sách record ở top-level.

Kết quả gồm thống kê tổng quan, percentile, phân bố theo khoảng độ dài, top segment dài/ngắn, thống kê theo file và biểu đồ. Phần path được thống kê riêng, không vẽ biểu đồ.


In [1]:
from pathlib import Path

# Đổi thành folder chứa processed context của bạn.
ROOT = Path('/mnt/mmlab2024nas/trantran/processed-contexts')

# Các field có thể chứa danh sách segment.
SEGMENT_FIELDS = [
    'passage_segments',
    'segments',
    'processed_context',
    'contexts',
]

SUPPORTED_EXTENSIONS = {'.json'}

ROOT = ROOT.expanduser().resolve()
print('Root:', ROOT)


Root: /mnt/mmlab2024nas/trantran/processed-contexts


In [2]:
import json
import numpy as np
import pandas as pd

def is_segment_list(value):
    return isinstance(value, list) and all(isinstance(x, str) for x in value)

def parse_segment(segment):
    """Parse a segment as: path|passage.

    If there is no '|', the segment has no path, so path='' and
    the whole segment is treated as the passage.
    """
    if '|' not in segment:
        return '', segment

    path, passage = segment.split('|', 1)
    return path.strip(), passage

def extract_segments(obj, source_file, record_idx=None, field_path=''):
    rows = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            path = f'{field_path}.{key}' if field_path else key

            if key in SEGMENT_FIELDS and is_segment_list(value):
                for seg_idx, segment in enumerate(value):
                    path_value, passage = parse_segment(segment)

                    rows.append({
                        'source_file': str(source_file),
                        'record_idx': record_idx,
                        'segment_idx': seg_idx,
                        'field': path,
                        'segment': segment,
                        'path': path_value,
                        'passage': passage,
                    })
            else:
                rows.extend(
                    extract_segments(value, source_file, record_idx, path)
                )

    elif isinstance(obj, list):
        for idx, item in enumerate(obj):
            rows.extend(
                extract_segments(
                    item,
                    source_file,
                    idx if record_idx is None else record_idx,
                    field_path
                )
            )

    return rows

def load_all_segments(root):
    rows = []
    json_files = sorted(
        p for p in root.rglob('*')
        if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
    )
    errors = []

    for path in json_files:
        try:
            with path.open('r', encoding='utf-8') as f:
                data = json.load(f)

            rows.extend(
                extract_segments(data, path.relative_to(root))
            )
        except Exception as e:
            errors.append({
                'source_file': str(path.relative_to(root)),
                'error': repr(e)
            })

    df = pd.DataFrame(rows)

    if not df.empty:
        # len() đếm Unicode code point -> phù hợp với "số ký tự".
        df['char_length'] = df['segment'].map(len)
        df['word_length'] = df['segment'].str.split().str.len()

        # Path luôn là phần trước dấu '|'. Nếu không có '|', path='' .
        df['path_char_length'] = df['path'].map(len)

        # Passage là phần sau dấu '|'; nếu không có '|', passage=toàn bộ segment.
        df['passage_char_length'] = df['passage'].map(len)
        df['passage_word_length'] = df['passage'].str.split().str.len()

    return df, pd.DataFrame(errors), json_files

df, errors_df, json_files = load_all_segments(ROOT)

print(f'Số file JSON: {len(json_files):,}')
print(f'Số segment:  {len(df):,}')
if not errors_df.empty:
    print(f'File lỗi:    {len(errors_df):,}')


Số file JSON: 8,532
Số segment:  212,656


In [3]:
if df.empty:
    raise ValueError(
        'Không tìm thấy segment. Hãy kiểm tra ROOT và SEGMENT_FIELDS.'
    )

display(df.head(10))
display(
    df[['char_length', 'path_char_length', 'passage_char_length', 'word_length', 'passage_word_length']]
    .describe(
        percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99]
    ).T
)


,source_file,record_idx,segment_idx,field,segment,path,passage,char_length,word_length,path_char_length,passage_char_length,passage_word_length
0,context_100050.json,0,0,passage_segments,| Mật độ sinh vật gây hại (con/m2) (áp dụng c...,,Mật độ sinh vật gây hại (con/m2) (áp dụng chu...,10093,2199,0,10091,2198
1,context_100062.json,0,0,passage_segments,Thong-tu-17-2022-TT-BGTVT-sua-doi-Thong-tu-12-...,Thong-tu-17-2022-TT-BGTVT-sua-doi-Thong-tu-12-...,BỘ GIAO THÔNG VẬN TẢI ------- CỘNG HÒA XÃ HỘI...,1192,243,92,1098,241
2,context_100062.json,0,1,passage_segments,Thong-tu-17-2022-TT-BGTVT-sua-doi-Thong-tu-12-...,Thong-tu-17-2022-TT-BGTVT-sua-doi-Thong-tu-12-...,- 1. Bổ sung điểm d khoản 4 Điều 6 như sau: “...,4630,1001,320,4308,946
3,context_100062.json,0,2,passage_segments,Thong-tu-17-2022-TT-BGTVT-sua-doi-Thong-tu-12-...,Thong-tu-17-2022-TT-BGTVT-sua-doi-Thong-tu-12-...,- 1. Thông tư này có hiệu lực thi hành kể từ ...,845,170,122,721,161
4,context_100109.json,0,0,passage_segments,Quyet-dinh-569-QD-TTg-2022-Chien-luoc-phat-tri...,Quyet-dinh-569-QD-TTg-2022-Chien-luoc-phat-tri...,THỦ TƯỚNG CHÍNH PHỦ ------- CỘNG HÒA XÃ HỘI C...,674,127,94,578,125
5,context_100109.json,0,1,passage_segments,Quyet-dinh-569-QD-TTg-2022-Chien-luoc-phat-tri...,Quyet-dinh-569-QD-TTg-2022-Chien-luoc-phat-tri...,"I. QUAN ĐIỂM PHÁT TRIỂN KHOA HỌC, CÔNG NGHỆ V...",54324,11784,243,54079,11749
6,context_100109.json,0,2,passage_segments,Quyet-dinh-569-QD-TTg-2022-Chien-luoc-phat-tri...,Quyet-dinh-569-QD-TTg-2022-Chien-luoc-phat-tri...,"- 1. Bộ Khoa học và Công nghệ - a) Chủ trì, x...",6767,1490,122,6643,1481
7,context_100109.json,0,3,passage_segments,Quyet-dinh-569-QD-TTg-2022-Chien-luoc-phat-tri...,Quyet-dinh-569-QD-TTg-2022-Chien-luoc-phat-tri...,"Các Bộ trưởng, Thủ trưởng cơ quan ngang bộ, T...",891,178,152,737,162
8,context_100125.json,0,0,passage_segments,Thong-tu-122-2021-TT-BTC-to-chuc-thuc-hien-du-...,Thong-tu-122-2021-TT-BTC-to-chuc-thuc-hien-du-...,BỘ TÀI CHÍNH ------- CỘNG HÒA XÃ HỘI CHỦ NGHĨ...,1200,240,81,1117,238
9,context_100125.json,0,1,passage_segments,Thong-tu-122-2021-TT-BTC-to-chuc-thuc-hien-du-...,Thong-tu-122-2021-TT-BTC-to-chuc-thuc-hien-du-...,"PHÂN CẤP NGUỒN THU, NHIỆM VỤ CHI VÀ PHÂN BỔ, ...",180,24,92,86,19


,count,mean,std,min,1%,5%,10%,25%,50%,75%,90%,95%,99%,max
char_length,212656.0,1610.372790,5173.888380,19.0,88.0,127.0,173.0,412.00,814.0,1518.0,2873.0,4519.00,14763.35,625806.0
path_char_length,212656.0,158.195325,112.306249,0.0,12.0,57.0,79.0,106.00,138.0,182.0,254.0,320.00,520.00,4365.0
passage_char_length,212656.0,1450.177465,5158.359909,3.0,15.0,21.0,52.0,250.75,650.0,1349.0,2695.0,4333.00,14627.45,625703.0
word_length,212656.0,341.612905,1126.024133,3.0,8.0,11.0,23.0,78.00,167.0,324.0,626.0,986.00,3224.00,138113.0
passage_word_length,212656.0,320.196801,1122.589223,1.0,3.0,4.0,12.0,55.00,145.0,301.0,601.0,960.25,3198.00,138104.0


In [4]:
length = df['char_length']

summary = pd.Series({
    'total_segments': len(df),
    'total_characters': int(length.sum()),
    'mean_chars': length.mean(),
    'median_chars': length.median(),
    'std_chars': length.std(),
    'min_chars': length.min(),
    'max_chars': length.max(),
    'p01_chars': length.quantile(.01),
    'p05_chars': length.quantile(.05),
    'p10_chars': length.quantile(.10),
    'p25_chars': length.quantile(.25),
    'p50_chars': length.quantile(.50),
    'p75_chars': length.quantile(.75),
    'p90_chars': length.quantile(.90),
    'p95_chars': length.quantile(.95),
    'p99_chars': length.quantile(.99),
})

display(summary.to_frame('value'))


,value
total_segments,2.126560e+05
total_characters,3.424554e+08
mean_chars,1.610373e+03
median_chars,8.140000e+02
std_chars,5.173888e+03
min_chars,1.900000e+01
max_chars,6.258060e+05
p01_chars,8.800000e+01
p05_chars,1.270000e+02
p10_chars,1.730000e+02


In [5]:
# Thống kê độ dài path (phần nằm trước dấu |). Nếu không có | thì path rỗng.
path_length = df['path_char_length']

path_summary = pd.Series({
    'total_segments': len(df),
    'total_path_characters': int(path_length.sum()),
    'mean_path_chars': path_length.mean(),
    'median_path_chars': path_length.median(),
    'std_path_chars': path_length.std(),
    'min_path_chars': path_length.min(),
    'max_path_chars': path_length.max(),
    'p01_path_chars': path_length.quantile(.01),
    'p05_path_chars': path_length.quantile(.05),
    'p10_path_chars': path_length.quantile(.10),
    'p25_path_chars': path_length.quantile(.25),
    'p50_path_chars': path_length.quantile(.50),
    'p75_path_chars': path_length.quantile(.75),
    'p90_path_chars': path_length.quantile(.90),
    'p95_path_chars': path_length.quantile(.95),
    'p99_path_chars': path_length.quantile(.99),
})

display(path_summary.to_frame('value'))


,value
total_segments,2.126560e+05
total_path_characters,3.364118e+07
mean_path_chars,1.581953e+02
median_path_chars,1.380000e+02
std_path_chars,1.123062e+02
min_path_chars,0.000000e+00
max_path_chars,4.365000e+03
p01_path_chars,1.200000e+01
p05_path_chars,5.700000e+01
p10_path_chars,7.900000e+01


In [6]:
# Thống kê độ dài passage (phần sau dấu |; nếu không có | thì là toàn bộ segment).
passage_length = df['passage_char_length']

passage_summary = pd.Series({
    'total_segments': len(df),
    'total_passage_characters': int(passage_length.sum()),
    'mean_passage_chars': passage_length.mean(),
    'median_passage_chars': passage_length.median(),
    'std_passage_chars': passage_length.std(),
    'min_passage_chars': passage_length.min(),
    'max_passage_chars': passage_length.max(),
    'p01_passage_chars': passage_length.quantile(.01),
    'p05_passage_chars': passage_length.quantile(.05),
    'p10_passage_chars': passage_length.quantile(.10),
    'p25_passage_chars': passage_length.quantile(.25),
    'p50_passage_chars': passage_length.quantile(.50),
    'p75_passage_chars': passage_length.quantile(.75),
    'p90_passage_chars': passage_length.quantile(.90),
    'p95_passage_chars': passage_length.quantile(.95),
    'p99_passage_chars': passage_length.quantile(.99),
})

display(passage_summary.to_frame('value'))


,value
total_segments,2.126560e+05
total_passage_characters,3.083889e+08
mean_passage_chars,1.450177e+03
median_passage_chars,6.500000e+02
std_passage_chars,5.158360e+03
min_passage_chars,3.000000e+00
max_passage_chars,6.257030e+05
p01_passage_chars,1.500000e+01
p05_passage_chars,2.100000e+01
p10_passage_chars,5.200000e+01


In [7]:
bins = [0, 100, 200, 300, 500, 750, 1000, 1500, 2000,
        3000, 5000, 10000, float('inf')]

labels = [
    '0–100', '101–200', '201–300', '301–500', '501–750',
    '751–1k', '1k–1.5k', '1.5k–2k', '2k–3k', '3k–5k',
    '5k–10k', '>10k'
]

df['length_bucket'] = pd.cut(
    df['char_length'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

bucket_stats = (
    df['length_bucket']
    .value_counts(sort=False)
    .rename('segments')
    .to_frame()
)

bucket_stats['percentage'] = bucket_stats['segments'] / len(df) * 100

display(bucket_stats)


,segments,percentage
length_bucket,,
0–100,3470,1.631743
101–200,21431,10.077778
201–300,12159,5.717685
301–500,28730,13.510082
501–750,33266,15.643104
751–1k,26111,12.278516
1k–1.5k,33501,15.753611
1.5k–2k,17690,8.318599
2k–3k,16428,7.725152


In [8]:
columns = [
    'source_file', 'record_idx', 'segment_idx', 'field',
    'char_length', 'word_length', 'path', 'passage', 'segment'
]

print('--- 20 segment dài nhất ---')
display(df.nlargest(20, 'char_length')[columns])

print('--- 20 segment ngắn nhất ---')
display(df.nsmallest(20, 'char_length')[columns])


--- 20 segment dài nhất ---


,source_file,record_idx,segment_idx,field,char_length,word_length,path,passage,segment
12911,context_118099.json,0,5,passage_segments,625806,138113,Quyet-dinh-27-2018-QD-TTg-ban-hanh-He-thong-ng...,"Bộ trưởng, Thủ trưởng cơ quan ngang bộ, Thủ t...",Quyet-dinh-27-2018-QD-TTg-ban-hanh-He-thong-ng...
76115,context_1992.json,0,3,passage_segments,616323,132966,Quyet-dinh-777-QD-CHK-2023-phe-duyet-tam-thoi-...,"Chánh Văn phòng Cục, Chánh Thanh tra Cục, các...",Quyet-dinh-777-QD-CHK-2023-phe-duyet-tam-thoi-...
107276,context_238624.json,0,21,passage_segments,296022,65944,Quyet-dinh-6858-QD-BYT-Bo-tieu-chi-chat-luong-...,Điểm chất lượng chung của bệnh viện được tính...,Quyet-dinh-6858-QD-BYT-Bo-tieu-chi-chat-luong-...
168232,context_42793.json,0,15,passage_segments,268598,59029,Thong-tu-08-2019-TT-BKHDT-huong-dan-ve-dinh-mu...,CpT = CchuẩnT x H1T x H2T x H3T x K1 Trong đó...,Thong-tu-08-2019-TT-BKHDT-huong-dan-ve-dinh-mu...
160548,context_32123.json,0,13,passage_segments,248947,57531,Thong-tu-25-2022-TT-BLDTBXH-che-do-trang-cap-p...,- 1. Thông tư này có hiệu lực kể từ ngày 01 t...,Thong-tu-25-2022-TT-BLDTBXH-che-do-trang-cap-p...
180846,context_58230.json,0,3,passage_segments,244732,55407,Quyet-dinh-1676-QD-BTC-2021-cong-bo-5-chuan-mu...,Vụ trưởng Vụ Tổ chức cán bộ; Vụ trưởng Vụ Phá...,Quyet-dinh-1676-QD-BTC-2021-cong-bo-5-chuan-mu...
136706,context_277700.json,0,2,passage_segments,237062,52767,Quyet-dinh-3191-QD-BCA-2022-cong-bo-thu-tuc-ha...,"Trường hợp nộp hồ sơ trực tuyến: văn bản, giấ...",Quyet-dinh-3191-QD-BCA-2022-cong-bo-thu-tuc-ha...
149710,context_293796.json,0,3,passage_segments,234175,52431,Quyet-dinh-1872-QD-BTP-2020-cong-bo-thu-tuc-ha...,NỘI DUNG CỤ THỂ CỦA CÁC THỦ TỤC HÀNH CHÍNH ĐƯ...,Quyet-dinh-1872-QD-BTP-2020-cong-bo-thu-tuc-ha...
93082,context_220347.json,0,3,passage_segments,228991,51072,Quyet-dinh-299-QD-BTP-thu-tuc-hanh-chinh-ban-h...,NỘI DUNG CỤ THỂ CỦA CÁC THỦ TỤC HÀNH CHÍNH BA...,Quyet-dinh-299-QD-BTP-thu-tuc-hanh-chinh-ban-h...
6114,context_108135.json,0,14,passage_segments,227999,52372,Thong-tu-04-2014-TT-BLDTBXH-huong-dan-thuc-hie...,- 1. Thông tư này có hiệu lực kể từ ngày 15 t...,Thong-tu-04-2014-TT-BLDTBXH-huong-dan-thuc-hie...


--- 20 segment ngắn nhất ---


,source_file,record_idx,segment_idx,field,char_length,word_length,path,passage,segment
14178,context_120070.json,0,0,passage_segments,19,5,,Hình vẽ minh họa,| Hình vẽ minh họa
44226,context_158798.json,0,145,passage_segments,22,6,12.3 Bếp,"Ngang - 0,8","12.3 Bếp | Ngang - 0,8"
100775,context_229300.json,0,0,passage_segments,22,6,,"1 Vị trí, kiến trúc","| 1 Vị trí, kiến trúc"
33689,context_144551.json,0,8,passage_segments,24,7,Chương II,SỞ HỮU NHÀ Ở,Chương II | SỞ HỮU NHÀ Ở
93067,context_2203.json,0,11,passage_segments,24,7,Điều 28,Điểm d khoản 1,Điều 28 | Điểm d khoản 1
133424,context_272778.json,0,2,passage_segments,24,6,0.1 g,L- ascorbic acid,0.1 g | L- ascorbic acid
133428,context_272778.json,0,6,passage_segments,24,6,0.1 g,L- ascorbic acid,0.1 g | L- ascorbic acid
3571,context_104343.json,0,4,passage_segments,25,6,Chương I,QUY ĐỊNH CHUNG,Chương I | QUY ĐỊNH CHUNG
4381,context_105570.json,0,1,passage_segments,25,6,Chương I,QUY ĐỊNH CHUNG,Chương I | QUY ĐỊNH CHUNG
9773,context_113369.json,0,1,passage_segments,25,6,Chương I,QUY ĐỊNH CHUNG,Chương I | QUY ĐỊNH CHUNG


In [9]:
file_stats = (
    df.groupby('source_file')
      .agg(
          segments=('char_length', 'size'),
          total_chars=('char_length', 'sum'),
          mean_chars=('char_length', 'mean'),
          median_chars=('char_length', 'median'),
          min_chars=('char_length', 'min'),
          max_chars=('char_length', 'max'),
          p95_chars=('char_length', lambda x: x.quantile(.95)),
          total_path_chars=('path_char_length', 'sum'),
          mean_path_chars=('path_char_length', 'mean'),
          median_path_chars=('path_char_length', 'median'),
          min_path_chars=('path_char_length', 'min'),
          max_path_chars=('path_char_length', 'max'),
          p95_path_chars=('path_char_length', lambda x: x.quantile(.95)),
      )
      .sort_values('segments', ascending=False)
)

display(file_stats)


,segments,total_chars,mean_chars,median_chars,min_chars,max_chars,p95_chars,total_path_chars,mean_path_chars,median_path_chars,min_path_chars,max_path_chars,p95_path_chars
source_file,,,,,,,,,,,,,
context_258607.json,925,360316,389.530811,215.0,129,20725,821.20,165713,179.149189,155.0,100,614,299.60
context_153054.json,839,435699,519.307509,458.0,139,1657,1024.50,71310,84.994041,79.0,34,211,130.10
context_132797.json,778,404846,520.367609,464.5,116,2145,1066.60,68541,88.098972,82.0,37,210,130.00
context_49846.json,472,167838,355.588983,325.5,157,1968,599.95,114406,242.385593,228.0,100,536,367.45
context_187506.json,463,318624,688.172786,603.0,77,4183,1541.10,56917,122.930886,120.0,46,262,189.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
context_249505.json,1,31407,31407.000000,31407.0,31407,31407,31407.00,0,0.000000,0.0,0,0,0.00
context_249608.json,1,4086,4086.000000,4086.0,4086,4086,4086.00,0,0.000000,0.0,0,0,0.00
context_29142.json,1,3349,3349.000000,3349.0,3349,3349,3349.00,0,0.000000,0.0,0,0,0.00


In [10]:
thresholds = [100, 200, 300, 500, 750, 1000, 1500, 2000, 3000, 5000]

threshold_stats = pd.DataFrame({
    'threshold_chars': thresholds,
    'segments_<=_threshold': [
        int((length <= t).sum()) for t in thresholds
    ],
})

threshold_stats['percentage'] = (
    threshold_stats['segments_<=_threshold'] / len(df) * 100
)

threshold_stats['segments_>_threshold'] = (
    len(df) - threshold_stats['segments_<=_threshold']
)

threshold_stats['percentage_>'] = 100 - threshold_stats['percentage']

display(threshold_stats)


,threshold_chars,segments_<=_threshold,percentage,segments_>_threshold,percentage_>
0,100,3470,1.631743,209186,98.368257
1,200,24901,11.709521,187755,88.290479
2,300,37060,17.427206,175596,82.572794
3,500,65790,30.937288,146866,69.062712
4,750,99056,46.580393,113600,53.419607
5,1000,125167,58.858908,87489,41.141092
6,1500,158668,74.612520,53988,25.387480
7,2000,176358,82.931119,36298,17.068881
8,3000,192786,90.656271,19870,9.343729
9,5000,203487,95.688342,9169,4.311658


In [11]:
OUTPUT_DIR = ROOT / '_segment_length_analysis'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(
    OUTPUT_DIR / 'segment_lengths.csv',
    index=False,
    encoding='utf-8-sig'
)

summary.to_csv(
    OUTPUT_DIR / 'summary.csv',
    header=True,
    encoding='utf-8-sig'
)

path_summary.to_csv(
    OUTPUT_DIR / 'path_summary.csv',
    header=True,
    encoding='utf-8-sig'
)

passage_summary.to_csv(
    OUTPUT_DIR / 'passage_summary.csv',
    header=True,
    encoding='utf-8-sig'
)

bucket_stats.to_csv(
    OUTPUT_DIR / 'length_buckets.csv',
    encoding='utf-8-sig'
)

file_stats.to_csv(
    OUTPUT_DIR / 'file_stats.csv',
    encoding='utf-8-sig'
)

threshold_stats.to_csv(
    OUTPUT_DIR / 'threshold_stats.csv',
    index=False,
    encoding='utf-8-sig'
)

if not errors_df.empty:
    errors_df.to_csv(
        OUTPUT_DIR / 'read_errors.csv',
        index=False,
        encoding='utf-8-sig'
    )

print('Đã export vào:', OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.iterdir()):
    print(' -', p.name)


Đã export vào: /mnt/mmlab2024nas/trantran/processed-contexts/_segment_length_analysis
 - file_stats.csv
 - length_buckets.csv
 - passage_summary.csv
 - path_summary.csv
 - segment_lengths.csv
 - summary.csv
 - threshold_stats.csv


## Cách đọc kết quả

- `path`: phần đứng trước dấu `|`; nếu segment không có `|` thì `path=''`.
- `passage`: phần đứng sau dấu `|`; nếu segment không có `|` thì `passage` là toàn bộ segment.
- `median_chars`: độ dài segment điển hình, ít bị ảnh hưởng bởi outlier hơn mean.
- `p95_chars` / `p99_chars`: hữu ích để chọn max length hoặc phát hiện segment bất thường.
- `length_buckets.csv`: tỷ lệ segment nằm trong từng khoảng ký tự.
- `segment_lengths.csv`: dữ liệu chi tiết từng segment để lọc/rà soát; có thêm `path` và `path_char_length`.
- `file_stats.csv`: kiểm tra file nào có segment/path quá dài hoặc phân bố khác biệt.
- `path_summary.csv`: thống kê tổng quan độ dài path (phần trước dấu `|`).

**Lưu ý:** `len(segment)` đếm số Unicode code point trong Python, phù hợp với cách hiểu thông thường về “số ký tự” tiếng Việt. Nếu cần đo byte UTF-8 hoặc grapheme cluster thì cách tính sẽ khác.
